In [2]:
import numpy as np
import matplotlib.pyplot as plt
import tifffile
import os
from patchify import patchify  #Only to handle large images
import random
from scipy import ndimage
import glob
import cv2
from tqdm import tqdm
import torch
import scipy
import torch.nn as nn

In [3]:
#Useful Functions
#Used to quickly display an image
def imshow(img, title, cmap = 'gray', axis = 'off', figsize = (10, 10)):
    plt.figure(figsize= figsize)
    plt.imshow(img, cmap = cmap)
    plt.title(title)
    plt.axis(axis)
    plt.show()  

def apply_multi_gabor(gray, orientations=8, ksize=21):
    """
    Apply multiple Gabor filters to enhance line structures in various directions.
    """
    filtered_imgs = []
    for theta in np.linspace(0, np.pi, orientations, endpoint=False):
        kernel = cv2.getGaborKernel((ksize, ksize), sigma=4.0, theta=theta,
                                    lambd=10.0, gamma=0.5, psi=0, ktype=cv2.CV_32F)
        filtered = cv2.filter2D(gray, cv2.CV_8UC1, kernel)
        filtered_imgs.append(filtered)
    return np.max(filtered_imgs, axis=0)

def preprocess_web(image, visualize = False):
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # Step 1: Morphological tophat
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3,5))
    tophat = cv2.morphologyEx(gray, cv2.MORPH_TOPHAT, kernel)

    # Step 2: Gabor filters for radial/circular lines
    gabor_enhanced = apply_multi_gabor(tophat, orientations=12, ksize=25)

    # Step 3: CLAHE
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    results = clahe.apply(gabor_enhanced)
    #Step 4: Binarize Web
    _, edited = cv2.threshold(results, 220, 255, cv2.THRESH_BINARY) 
    #Step 5: Display Results if visualize is True
    if visualize:
        imshow(edited, title = "Results")
    #Step 6: Return preprocessed image of the web
    return edited

In [5]:
#Storing all file paths of images in a list of strings
import os
directory = r"C:\Users\adamain\Downloads\ProcessedSpiderWebs"
image_paths = []
for filename in os.listdir(directory):
    if filename.lower().endswith(".png"):
        image_paths.append(os.path.join(directory, filename))
print(image_paths)

['C:\\Users\\adamain\\Downloads\\ProcessedSpiderWebs\\cropped_19 - 1_web.png', 'C:\\Users\\adamain\\Downloads\\ProcessedSpiderWebs\\cropped_19 - 2_web.png', 'C:\\Users\\adamain\\Downloads\\ProcessedSpiderWebs\\cropped_19 - 3_web.png', 'C:\\Users\\adamain\\Downloads\\ProcessedSpiderWebs\\cropped_19 - C_web.png', 'C:\\Users\\adamain\\Downloads\\ProcessedSpiderWebs\\cropped_25 - 1_web.png', 'C:\\Users\\adamain\\Downloads\\ProcessedSpiderWebs\\cropped_25 - 2_web.png', 'C:\\Users\\adamain\\Downloads\\ProcessedSpiderWebs\\cropped_25 - 3_web.png', 'C:\\Users\\adamain\\Downloads\\ProcessedSpiderWebs\\cropped_25 - C_web.png', 'C:\\Users\\adamain\\Downloads\\ProcessedSpiderWebs\\cropped_26 - 1_web.png', 'C:\\Users\\adamain\\Downloads\\ProcessedSpiderWebs\\cropped_26 - 2_web.png', 'C:\\Users\\adamain\\Downloads\\ProcessedSpiderWebs\\cropped_26 - 3_web.png', 'C:\\Users\\adamain\\Downloads\\ProcessedSpiderWebs\\cropped_26 - 4_web.png', 'C:\\Users\\adamain\\Downloads\\ProcessedSpiderWebs\\cropped_26

In [7]:
#Preprocess All Images
preprocessed_images = []
for path in image_paths:
    image = cv2.imread(path)
    preprocessed_images.append(preprocess_web(image))

In [6]:
#Get center coordinates of web hub for all images
coordinates = []
with open("web_centers.txt", "r") as f:
    for line in f:
        x, y = map(int, line.strip().split(","))
        coordinates.append((x, y))


In [38]:
import numpy as np
from skimage.transform import rotate as sk_rotate
from itertools import combinations
import math

def enhanced_symmetry_scoring(images, centers, rotation_steps=36, verbose=False):
    """
    Enhanced version with logarithmic scaling to widen the score range.
    Returns both raw and scaled scores (0-1 range where 1 = perfect symmetry).
    """
    
    results = []
    scaled_scores = []
    
    for idx, (img, center) in enumerate(zip(images, centers)):
        if verbose:
            print(f"Processing image {idx+1}/{len(images)}")
            
        # Initialize result storage
        result = {
            'raw_pairwise_scores': [],
            'raw_axial_scores': [],
            'final_raw_score': 0,
            'final_scaled_score': 0
        }
        
        # Generate rotated versions
        rotated_images = []
        for angle in np.linspace(0, 360, rotation_steps, endpoint=False):
            rotated = sk_rotate(img, angle, center=center, order=0, mode='constant', cval=0)
            rotated = (rotated > 0.5).astype(np.uint8)
            rotated_images.append(rotated)
        
        # 1. Pairwise rotational comparisons
        for (i, img1), (j, img2) in combinations(enumerate(rotated_images), 2):
            intersection = np.logical_and(img1, img2)
            union = np.logical_or(img1, img2)
            if np.sum(union) > 0:
                score = np.sum(intersection) / np.sum(union)
                result['raw_pairwise_scores'].append(score)
        
        # 2. Axial symmetry checks
        for angle in [0, 45, 90, 135]:
            mirrored = sk_rotate(img, 2*angle, center=center, order=0)
            mirrored = np.flip(mirrored, axis=1)
            mirrored = sk_rotate(mirrored, -2*angle, center=center, order=0)
            mirrored = (mirrored > 0.5).astype(np.uint8)
            
            intersection = np.logical_and(img, mirrored)
            union = np.logical_or(img, mirrored)
            if np.sum(union) > 0:
                score = np.sum(intersection) / np.sum(union)
                result['raw_axial_scores'].append(score)
        
        # Calculate composite raw score (weighted average)
        pairwise_mean = np.mean(result['raw_pairwise_scores']) if result['raw_pairwise_scores'] else 0
        axial_mean = np.mean(result['raw_axial_scores']) if result['raw_axial_scores'] else 0
        raw_score = 0.6*pairwise_mean + 0.4*axial_mean  # Slightly favor rotational symmetry
        result['final_raw_score'] = raw_score
        
        # Apply logarithmic scaling to widen the range
        if raw_score > 0:
            # Logarithmic transformation parameters tuned for your 0.16-0.22 range
            log_score = math.log10(raw_score * 4 + 1)  # Scale then log
            # Normalize to 0-1 range based on expected maximum
            scaled_score = min(1.0, max(0.0, log_score / math.log10(1.8)))
        else:
            scaled_score = 0.0
        
        result['final_scaled_score'] = scaled_score
        results.append(result)
        scaled_scores.append(scaled_score)
    
    return results, scaled_scores

In [41]:
results, scaled_scores = enhanced_symmetry_scoring(preprocessed_images, coordinates, rotation_steps=28, verbose=True) #Going off of calculated averages- rotation_steps = 28. default is 36

Processing image 1/69
Processing image 2/69
Processing image 3/69
Processing image 4/69
Processing image 5/69
Processing image 6/69
Processing image 7/69
Processing image 8/69
Processing image 9/69
Processing image 10/69
Processing image 11/69
Processing image 12/69
Processing image 13/69
Processing image 14/69
Processing image 15/69
Processing image 16/69
Processing image 17/69
Processing image 18/69


KeyboardInterrupt: 

In [42]:
# Print all scores
print("Scaled symmetry scores (0-1):")
for i, score in enumerate(scaled_scores):
    print(f"Web {i+1}: {score:.3f}")


Scaled symmetry scores (0-1):
Web 1: 1.000
Web 2: 0.843
